# Определение оптимального размера выборки методом Монте-Карло

**Контекст:** К нам пришли наши коллеги из ML-отдела и рассказали, что планируют выкатывать новый алгоритм, рекомендующий нашим пользователям интересные посты. После обсуждений того, как он это делает, вы пришли к следующему пониманию:

Алгоритм добавляет пользователям 1-2 просмотра
Вероятность того, что он сработает, составляет 90%
Если у пользователя меньше 50 просмотров, то алгоритм не сработает.

В рамках оценки нового алгоритма рекомендаций ленты постов нам необходимо определить минимальный размер выборки ($N$) на одну группу (тест/контроль). Мы хотим зафиксировать стандартные параметры статистической мощности и значимости, чтобы гарантированно обнаружить ожидаемый эффект.

### Параметры эксперимента
* **Базовая метрика (Контроль):** CTR из просмотров в лайки.
* **Минимально детектируемый эффект (MDE):** +1-2 просмотра.
* **Уровень значимости ($\alpha$):** 0.05 (вероятность ошибки I рода).
* **Целевая мощность ($1 - \beta$):** 0.80 (вероятность обнаружить реальный эффект в 80% случаев).

### Логика метода Монте-Карло
Поскольку аналитические формулы (t-test power formulas) строго привязаны к идеальному нормальному распределению, мы применим симуляцию Монте-Карло для более гибкого расчета. 
Для каждого тестируемого размера выборки $N$ мы:
1. Имитируем $M = 20000$ случайных экспериментов (генерация выборок из распределения с эффектом и без).
2. Для каждого эксперимента рассчитываем $p$-value с помощью двухвыборочного t-теста.
3. Считаем долю экспериментов, где $p < \alpha$. Эта доля и есть эмпирическая мощность модели для данного размера выборки.
4. Выбираем минимальное $N$, при котором мощность достигает $\ge 80\%$.


In [ ]:
import pandas as pd
import pandahouse
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from scipy.stats import ttest_ind
import numpy as np
from scipy import stats

In [2]:
rng = np.random.default_rng()

In [3]:
# Параметры подключения
connection = {
    'host': 'http://clickhouse.lab.karpov.courses:8123',
    'password': 'dpo_python_2020',
    'user': 'student',
    'database': 'simulator_20260420'
}

In [4]:
# Достанем распределение просмотров

In [5]:
q = """
select views, 
    count() as users
from (select user_id,
    sum(action = 'view') as views
    from {db}.feed_actions
    where toDate(time) between '2026-03-27' and '2026-04-02'
    group by user_id
)
group by views
order by views
"""

In [6]:
views_distribution = pandahouse.read_clickhouse(q, connection=connection)

In [7]:
# Вычислим общее число пользователей за период
count_of_users = views_distribution['users'].sum()

In [8]:
views_distribution['p'] = views_distribution['users']/count_of_users
views_distribution.sort_values(by = 'p', ascending = False)

,views,users,p
15,16,545,0.012977
14,15,537,0.012787
13,14,500,0.011906
34,35,485,0.011548
29,30,469,0.011167
...,...,...,...
280,287,1,0.000024
278,285,1,0.000024
276,280,1,0.000024
1,2,1,0.000024


In [9]:
print(f'Общее число пользователей в исторический период: {count_of_users}')

Общее число пользователей в исторический период: 41997


In [10]:
# Достанем распределение CTR

In [11]:
q_1 = """
select round(ctr, 2) as ctr, 
   count(user_id) as users
from (select toDate(time) as dt,
    user_id,
    sum(action = 'like')/sum(action = 'view') as ctr
from {db}.feed_actions
where toDate(time) between '2026-03-27' and '2026-04-02'
group by dt, user_id
)
group by ctr
"""

In [12]:
ctr_distribution = pandahouse.read_clickhouse(q_1, connection=connection)

In [13]:
ctr_distribution['p'] = ctr_distribution['users']/ctr_distribution['users'].sum()
ctr_distribution.sort_values(by = 'p', ascending = False)

,ctr,users,p
29,0.17,4291,0.050411
7,0.18,4177,0.049071
73,0.21,4125,0.048460
18,0.20,4059,0.047685
55,0.19,4029,0.047333
...,...,...,...
69,0.89,1,0.000012
8,0.72,1,0.000012
74,0.83,1,0.000012
58,0.76,1,0.000012


In [14]:
# Генерируем ctr
def get_ctrs(x, y, p_array, values_array):
    return rng.choice(values_array, size=(x, y), p=p_array).astype("float32")

In [21]:
n_experiments = 20000  # Количество симуляций (не меньше 20000)
n_users = 55000       # Количество пользователей в одной симуляции

In [22]:
# Нормализуем вероятности 
p_views_norm = views_distribution['p'] / np.sum(views_distribution['p'])
p_ctr_norm = ctr_distribution['p'] / np.sum(ctr_distribution['p'])

In [23]:
# Список для сохранения результатов 
p_values = []

In [24]:
for _ in tqdm(range(n_experiments)):
    # 1. Генерируем просмотры для группы B и применяем эффект для группы A
    group_A_views = rng.choice(views_distribution['views'], size=n_users, p=p_views_norm).astype(int)
    
    group_B_views = rng.choice(views_distribution['views'], size=n_users, p=p_views_norm).astype(int)
    group_B_views = group_B_views + ((1 + rng.binomial(n=1, p=0.5, size=n_users)) * rng.binomial(n=1, p=0.9, size=n_users) * (group_B_views >= 30)).astype(int)
    
    # 2. Генерируем CTR 
    ctr_A = rng.choice(ctr_distribution['ctr'], size=n_users, p=p_ctr_norm)
    ctr_B = rng.choice(ctr_distribution['ctr'], size=n_users, p=p_ctr_norm)
    
    # 3. Генерация кликов
    clicks_A = rng.binomial(n=group_A_views, p=np.clip(ctr_A, 0, 1))
    clicks_B = rng.binomial(n=group_B_views, p=np.clip(ctr_B, 0, 1))
    
    # 4. Применение t-критерия Уэлча и запись p-value в список
    p_val = stats.ttest_ind(clicks_A, clicks_B, equal_var=False).pvalue
    p_values.append(p_val)

100%|██████████| 20000/20000 [19:08<00:00, 17.41it/s] 


In [25]:
p_values = np.array(p_values)

In [26]:
beta = np.sum(p_values <= 0.05) / n_experiments
print(f"Мощность критерия: {beta:.4f}")

Мощность критерия: 0.8139


Вывод: 
- При размере выборки 11000 пользователей мы достигнем целевой мощности.
- Отталкиваяь от исторических данных недельной давности, у нас не получится набрать необходимо число пользователей за одну неделю эксперимента.